In [0]:
create database retail_sales_catalog.sales;

In [0]:
create volume retail_sales_catalog.sales.data;

In [0]:
%python

# reading retails csv dataset
raw_orders = spark.read.csv("/Volumes/retail_sales_catalog/sales/data/bronze/retail_dataset.csv", header=True, inferSchema=True)
raw_orders.show()

# reading retails json dataset
raw_customers = spark.read.json('/Volumes/retail_sales_catalog/sales/data/bronze/customer_dataset.json')
raw_customers.show()

In [0]:
%python

from pyspark.sql.functions import *

spark.conf.set("spark.sql.ansi.enabled", "false")

order_ts = coalesce(
    to_timestamp(trim(col("order_date")), "yyyy-MM-dd"),
    to_timestamp(trim(col("order_date")), "dd/MM/yyyy"),
    to_timestamp(trim(col("order_date")), "d-MMM-yyyy"),
    to_timestamp(trim(col("order_date")), "MM-dd-yyyy"),
    to_timestamp(trim(col("order_date")), "dd-MM-yyyy")
)

clean_df = (
    raw_orders
    .withColumn("order_id", trim(col("order_id")).cast("long"))
    .withColumn("order_date_ts", order_ts)
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name", lower(trim(col("product_name"))))
    .withColumn("category", lower(trim(col("category"))))
    .withColumn("quantity", when(trim(col("quantity")).isNull() | (trim(col("quantity"))==""), lit(1)).otherwise(trim(col("quantity"))))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("quantity", when(col("quantity") <= 0, lit(1)).otherwise(col("quantity")))
    .withColumn("price_clean", translate(trim(col("price")), "$,", ""))
    .withColumn("price", col("price_clean").cast("double"))
    .drop("price_clean")
    .withColumn("payment_type", lower(trim(col("payment_type"))))
    .withColumn("order_status", lower(trim(col("order_status"))))
    .withColumn("returned", lower(trim(col("returned"))))
    .withColumn("total_amount", (coalesce(col("quantity"), lit(0)) * coalesce(col("price"), lit(0.0))).cast("double"))
)

clean_df = clean_df.dropDuplicates(["order_id", "product_id"]).filter(col("order_id").isNotNull() & col("product_id").isNotNull())
clean_df.write.format("delta").mode("overwrite").saveAsTable("retail_sales_catalog.sales.silver_orders")


In [0]:
select * from retail_sales_catalog.sales.silver_orders;

In [0]:
%python
# ---------- SILVER: Customers ----------

signup_ts = coalesce(
    to_timestamp(trim(col("signup_date")), "yyyy-MM-dd"),
    to_timestamp(trim(col("signup_date")), "dd/MM/yyyy"),
    to_timestamp(trim(col("signup_date")), "d-MMM-yyyy"),
    to_timestamp(trim(col("signup_date")), "yyyy/MM/dd")
)

clean_customers = (
    raw_customers
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("gender", lower(trim(col("gender"))))
    .withColumn("age", when(trim(col("age")).isNull() | (trim(col("age"))==""), None).otherwise(col("age")))
    .withColumn("age", when(col("age").cast('int') < 0, None).otherwise(col("age").cast('int')))
    .withColumn("city", trim(col("city")))
    .withColumn("loyalty_tier", lower(trim(col("loyalty_tier"))))
    .withColumn("signup_date", signup_ts)
)

#display(clean_customers)

clean_customers.write.format("delta").mode("overwrite").saveAsTable("retail_sales_catalog.sales.silver_customers")

In [0]:
select * from retail_sales_catalog.sales.silver_customers;

In [0]:
%python

# GOLD: aggregate (product x customer x year x month x date)
silver_orders = spark.table("retail_sales_catalog.sales.silver_orders")
silver_customers = spark.table("retail_sales_catalog.sales.silver_customers")

orders_enriched = silver_orders.join(silver_customers, on="customer_id", how="left")

orders_enriched = orders_enriched   \
            .withColumn("order_date", to_date(col("order_date_ts")))    \
            .withColumn("year", year(col("order_date_ts"))) \
            .withColumn("month", month(col("order_date_ts")))

gold_df = (
    orders_enriched
    .groupBy("product_id","product_name","customer_id","category","year","month","order_date", "gender", "age", "city", "loyalty_tier")
    .agg(
        sum("total_amount").alias("total_sales"),
        sum(coalesce(col("quantity"), lit(0))).alias("total_quantity"),
        countDistinct("order_id").alias("total_orders"),
        min("price").alias("min_price"),
        max("price").alias("max_price"),
        avg("price").alias("avg_unit_price"),
        min("order_date_ts").alias("first_order_date"),
        max("order_date_ts").alias("last_order_date"),
        sum(when(col("returned") == "yes", 1).otherwise(0)).alias("returned_count"),
        sum(when(col("returned") == "yes", col("total_amount")).otherwise(0.0)).alias("returned_amount")
        # countDistinct("payment_type").alias("distinct_payment_types"),
        # first(col("customer_name"), True).alias("customer_name")
    )
)

gold_df = gold_df.withColumn("avg_order_value", when(col("total_orders")>0, col("total_sales")/col("total_orders")).otherwise(lit(0.0))) \
                 .withColumn("avg_price_per_item", when(col("total_quantity")>0, col("total_sales")/col("total_quantity")).otherwise(lit(0.0))) \
                 .withColumn("days_since_last_order", when(col("last_order_date").isNotNull(), datediff(current_date(), col("last_order_date"))).otherwise(lit(None))) \
                 .withColumn("refreshed_at", current_timestamp())

#display(gold_df)

gold_df.write.format("delta").mode("overwrite").saveAsTable("retail_sales_catalog.sales.golden_records")


In [0]:
select * from retail_sales_catalog.sales.golden_records;

In [0]:
%python
# # KPIs (from silver to avoid double-counting)
# overall = spark.table("retail_demo.silver_orders").agg(
#     _sum("total_amount").alias("total_revenue"),
#     countDistinct("order_id").alias("total_orders"),
#     _sum(coalesce(col("quantity"), lit(0))).alias("total_items"),
#     _sum(when(col("returned")=="yes", col("total_amount")).otherwise(0.0)).alias("total_returned_amount"),
#     countDistinct("customer_id").alias("total_customers")
# ).collect()[0]

# total_revenue = float(overall["total_revenue"] or 0.0)
# total_orders = int(overall["total_orders"] or 0)
# total_items = int(overall["total_items"] or 0)
# total_returned_amount = float(overall["total_returned_amount"] or 0.0)
# total_customers = int(overall["total_customers"] or 0)

# avg_order_value = (total_revenue / total_orders) if total_orders>0 else 0.0
# avg_price_per_item = (total_revenue / total_items) if total_items>0 else 0.0
# avg_items_per_order = (total_items / total_orders) if total_orders>0 else 0.0

# kpi_row = [{
#     "total_revenue": total_revenue,
#     "total_orders": total_orders,
#     "total_items": total_items,
#     "total_customers": total_customers,
#     "avg_order_value": avg_order_value,
#     "avg_price_per_item": avg_price_per_item,
#     "avg_items_per_order": avg_items_per_order,
#     "refreshed_at": current_timestamp()
# }]

# spark.createDataFrame(kpi_row).write.format("delta").mode("overwrite").saveAsTable("retail_demo.gold_kpis_final")

# print("Pipeline finished. Tables created: retail_demo.bronze_orders, retail_demo.silver_orders, retail_demo.gold_aggregates, retail_demo.gold_kpis_final")
